# Notebook 14: Feature Steering as Active Defense

**Purpose:** Demonstrate the novel contribution of IRIS Phase 2 — using causal interventions to *neutralize* injections at the representation level, not just detect and block them.

**Key insight:** We use the SAE's decoder to map modified feature vectors back to the residual stream, suppressing injection-sensitive features while preserving normal representation. This is analogous to a network IPS rewriting malicious packets rather than just dropping them.

**Experiments:**
1. Steering accuracy: suppress top-20 features, measure classification flip rate
2. Steering fidelity: verify normal prompts are unaffected
3. Strategy comparison: block-on-detect vs steer-then-classify vs steer-then-pass
4. Adaptive steering: calibrate dampening scale based on threat probability

**Prerequisites:** Run notebooks 10-12 first.

**Runtime:** Colab GPU (T4), ~25 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q
!pip install -q "numpy<2.0"

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/iris')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

DRIVE_ROOT = Path('/content/drive/MyDrive/iris')

## Step 0: Load Models and Data

In [ ]:
from src.model.transformer import load_model
from src.sae.architecture import SparseAutoencoder
from src.utils.helpers import load_checkpoint
from src.data.dataset import IrisDataset
from sklearn.linear_model import LogisticRegression

# Load GPT-2
gpt2 = load_model(device)

# Load SAE
sae = SparseAutoencoder(d_input=1280, expansion_factor=8, sparsity_coeff=1e-4)
ckpt_path = DRIVE_ROOT / 'checkpoints' / 'sae_d10240_lambda1e-04.pt'
load_checkpoint(ckpt_path, sae, device=device)
sae = sae.to(device).eval()

# Load dataset and features
ds_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_balanced.json'
expanded_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_expanded.json'
if expanded_path.exists():
    ds_path = expanded_path
dataset = IrisDataset.load(ds_path)
labels = np.array(dataset.labels)

sensitivity = np.load(str(DRIVE_ROOT / 'checkpoints' / 'sensitivity_scores.npy'))
feature_matrix = np.load(str(DRIVE_ROOT / 'checkpoints' / 'feature_matrix.npy'))

# Train detector
detector = LogisticRegression(max_iter=1000, random_state=42)
detector.fit(feature_matrix, labels)
print(f'Detector accuracy: {detector.score(feature_matrix, labels):.3f}')
print(f'Dataset: {len(dataset)} examples ({sum(labels)} injections, {len(labels) - sum(labels)} normal)')

## Step 1: Initialize Steering Defense

In [ ]:
from src.agent.steering import SteeringDefense

steering = SteeringDefense(
    sae_model=sae,
    sensitivity_scores=sensitivity,
    gpt2_model=gpt2,
    detector=detector,
    top_k=20,
    layer=35,
)

print(f'Steering defense initialized')
print(f'Top-20 injection features: {steering.injection_feature_indices}')
print(f'Their sensitivities: {sensitivity[steering.injection_feature_indices][:5]}...')

## Experiment 1: Steering Accuracy on Injections

For each injection prompt, suppress top-20 injection-sensitive features (scale=0.0) and measure whether the classification flips from injection to normal.

In [ ]:
# Sample injection and normal prompts
inject_idx = np.where(labels == 1)[0]
normal_idx = np.where(labels == 0)[0]
np.random.seed(42)

n_test = 30  # Keep manageable for Colab runtime
sample_inject = np.random.choice(inject_idx, size=min(n_test, len(inject_idx)), replace=False)
sample_normal = np.random.choice(normal_idx, size=min(n_test, len(normal_idx)), replace=False)

inject_texts = [dataset.texts[i] for i in sample_inject]
normal_texts = [dataset.texts[i] for i in sample_normal]

# Steer injections
print(f'Steering {len(inject_texts)} injection prompts (scale=0.0)...\n')
inject_results = steering.batch_dampen(inject_texts, scale=0.0)

n_flips = sum(1 for r in inject_results if r['flip'])
prob_drops = [r['orig_prob'] - r['steered_prob'] for r in inject_results]

print(f'Results:')
print(f'  Classification flips: {n_flips}/{len(inject_results)} ({n_flips/len(inject_results):.0%})')
print(f'  Mean probability drop: {np.mean(prob_drops):.3f}')
print(f'  Mean orig probability: {np.mean([r["orig_prob"] for r in inject_results]):.3f}')
print(f'  Mean steered probability: {np.mean([r["steered_prob"] for r in inject_results]):.3f}')

## Experiment 2: Steering Fidelity on Normal Prompts

Verify that steering does not affect normal prompts (their injection features should already be near-zero).

In [ ]:
print(f'Steering {len(normal_texts)} normal prompts (scale=0.0)...\n')
normal_results = steering.batch_dampen(normal_texts, scale=0.0)

n_normal_flips = sum(1 for r in normal_results if r['flip'])
normal_changes = [abs(r['orig_prob'] - r['steered_prob']) for r in normal_results]

print(f'Fidelity results:')
print(f'  Classification flips: {n_normal_flips}/{len(normal_results)} ({n_normal_flips/len(normal_results):.0%})')
print(f'  Mean probability change: {np.mean(normal_changes):.4f}')
print(f'  Max probability change: {np.max(normal_changes):.4f}')
print(f'  Mean orig probability: {np.mean([r["orig_prob"] for r in normal_results]):.3f}')
print(f'  Mean steered probability: {np.mean([r["steered_prob"] for r in normal_results]):.3f}')
print()
if n_normal_flips == 0:
    print('Perfect fidelity: no normal prompts were affected by steering.')
else:
    print(f'WARNING: {n_normal_flips} normal prompts were affected.')

## Experiment 3: Defense Strategy Comparison

Compare three strategies:
- **(a) Block-on-detect:** Block if probability > threshold (current approach)
- **(b) Steer-then-classify:** Apply steering, then re-classify. Block only if still flagged.
- **(c) Steer-then-pass:** Apply steering and always pass (most permissive)

We measure false positive rate (normal prompts incorrectly blocked) and attack survival (injections that get through).

In [ ]:
threshold = 0.5

# Strategy (a): Block-on-detect
a_blocked_inject = sum(1 for r in inject_results if r['orig_prob'] >= threshold)
a_blocked_normal = sum(1 for r in normal_results if r['orig_prob'] >= threshold)

# Strategy (b): Steer-then-classify
b_blocked_inject = sum(1 for r in inject_results if r['steered_prob'] >= threshold)
b_blocked_normal = sum(1 for r in normal_results if r['steered_prob'] >= threshold)

# Strategy (c): Steer-then-pass (never blocks, but injection signal neutralized)
c_survival = sum(1 for r in inject_results if r['steered_prob'] >= threshold)

n_inj = len(inject_results)
n_nor = len(normal_results)

print('Defense Strategy Comparison')
print('=' * 60)
print(f'{"Strategy":<25} {"Det Rate":<12} {"FPR":<12} {"Survival":<12}')
print('-' * 60)
print(f'{"(a) Block-on-detect":<25} {a_blocked_inject/n_inj:<12.1%} {a_blocked_normal/n_nor:<12.1%} {(n_inj-a_blocked_inject)/n_inj:<12.1%}')
print(f'{"(b) Steer-then-classify":<25} {b_blocked_inject/n_inj:<12.1%} {b_blocked_normal/n_nor:<12.1%} {(n_inj-b_blocked_inject)/n_inj:<12.1%}')
print(f'{"(c) Steer-then-pass":<25} {"N/A":<12} {"0.0%":<12} {c_survival/n_inj:<12.1%}')
print('=' * 60)

## Experiment 4: Adaptive Steering (Dose-Response)

Calibrate the dampening scale based on the detected threat probability. Higher threat → stronger suppression.

In [ ]:
# Test adaptive steering on injection prompts
print('Adaptive steering on injection prompts:\n')
adaptive_results = []
for i, text in enumerate(inject_texts[:15]):
    # First get the detection probability
    orig_prob = inject_results[i]['orig_prob']
    result = steering.adaptive_dampen(text, probability=orig_prob)
    adaptive_results.append(result)
    
    if i < 5:  # Show first 5
        print(f'  [{i}] orig={result["orig_prob"]:.3f} -> steered={result["steered_prob"]:.3f} '
              f'(scale={result["scale"]:.2f}, flip={result["flip"]})')

n_adaptive_flips = sum(1 for r in adaptive_results if r['flip'])
print(f'\nAdaptive flips: {n_adaptive_flips}/{len(adaptive_results)} ({n_adaptive_flips/len(adaptive_results):.0%})')

In [ ]:
# Dose-response curve: vary scale from 0.0 to 2.0
dose_scales = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.25, 1.5, 1.75, 2.0]

# Use subset for speed
dose_inject = inject_texts[:10]
dose_normal = normal_texts[:10]

inject_dose_probs = []
normal_dose_probs = []

for scale in dose_scales:
    inj_res = steering.batch_dampen(dose_inject, scale=scale)
    nor_res = steering.batch_dampen(dose_normal, scale=scale)
    inject_dose_probs.append(np.mean([r['steered_prob'] for r in inj_res]))
    normal_dose_probs.append(np.mean([r['steered_prob'] for r in nor_res]))
    print(f'Scale {scale:.2f}: inject={inject_dose_probs[-1]:.3f}, normal={normal_dose_probs[-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(dose_scales, inject_dose_probs, 'o-', color='#D55E00', linewidth=2,
        markersize=6, label='Injection prompts')
ax.plot(dose_scales, normal_dose_probs, 's-', color='#0072B2', linewidth=2,
        markersize=6, label='Normal prompts')

ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Decision boundary')
ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.text(1.02, 0.02, 'No intervention', rotation=90, transform=ax.get_xaxis_transform(),
        fontsize=9, color='gray', va='bottom')

ax.set_xlabel('Feature Scale Factor', fontsize=12)
ax.set_ylabel('Mean Detector Probability (Injection)', fontsize=12)
ax.set_title('Feature Steering Dose-Response \u2014 Defense Calibration', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(-0.05, 2.1)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
save_path = DRIVE_ROOT / 'results' / 'figures' / 'steering_dose_response.png'
fig.savefig(str(save_path), dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved to {save_path}')

In [ ]:
# Visualize feature changes for a single injection prompt
example_result = inject_results[0]
orig_feats = example_result['orig_features']
steered_feats = example_result['steered_features']

# Show top-20 injection features
top_20_idx = steering.injection_feature_indices

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: before steering
ax = axes[0]
ax.bar(range(20), orig_feats[top_20_idx], color='#D55E00', alpha=0.85)
ax.set_xlabel('Feature Rank (by sensitivity)', fontsize=11)
ax.set_ylabel('Activation', fontsize=11)
ax.set_title(f'Before Steering (prob={example_result["orig_prob"]:.3f})', fontsize=12)

# Right: after steering
ax = axes[1]
ax.bar(range(20), steered_feats[top_20_idx], color='#0072B2', alpha=0.85)
ax.set_xlabel('Feature Rank (by sensitivity)', fontsize=11)
ax.set_ylabel('Activation', fontsize=11)
ax.set_title(f'After Steering (prob={example_result["steered_prob"]:.3f})', fontsize=12)

# Match y-axis scales
max_y = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1])
axes[0].set_ylim(0, max_y)
axes[1].set_ylim(0, max_y)

plt.tight_layout()
save_path = DRIVE_ROOT / 'results' / 'figures' / 'steering_before_after.png'
fig.savefig(str(save_path), dpi=200, bbox_inches='tight')
plt.show()

## Results Summary & Interpretation

In [ ]:
steering_results_data = {
    'experiment': 'feature_steering_defense',
    'top_k': 20,
    'n_injection_tested': len(inject_texts),
    'n_normal_tested': len(normal_texts),
    'injection_flip_rate': n_flips / len(inject_results),
    'mean_prob_drop': float(np.mean(prob_drops)),
    'normal_flip_rate': n_normal_flips / len(normal_results),
    'mean_normal_change': float(np.mean(normal_changes)),
    'strategy_comparison': {
        'block_on_detect': {
            'detection_rate': a_blocked_inject / n_inj,
            'fpr': a_blocked_normal / n_nor,
        },
        'steer_then_classify': {
            'detection_rate': b_blocked_inject / n_inj,
            'fpr': b_blocked_normal / n_nor,
        },
    },
    'dose_response': {
        'scales': dose_scales,
        'injection_probs': inject_dose_probs,
        'normal_probs': normal_dose_probs,
    },
}

results_path = DRIVE_ROOT / 'results' / 'metrics' / 'feature_steering_defense.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(steering_results_data, indent=2))
print(f'Results saved to {results_path}')

print()
print('=' * 60)
print('  Feature Steering Defense \u2014 Key Findings')
print('=' * 60)
print(f'  Injection flip rate:  {n_flips}/{len(inject_results)} ({n_flips/len(inject_results):.0%})')
print(f'  Normal fidelity:      {len(normal_results)-n_normal_flips}/{len(normal_results)} ({(len(normal_results)-n_normal_flips)/len(normal_results):.0%})')
print(f'  Mean prob drop:       {np.mean(prob_drops):.3f}')
print()
print('This demonstrates that feature steering can neutralize')
print('injections at the representation level while preserving')
print('normal prompt processing \u2014 a novel defense mechanism.')